# 2. Ringing a black hole and extracting gravitational waves

A Schwarzschild black hole ($M=1$) in Kerr-Schild coordinates, with an $\ell=2, m=0$ perturbation added to the metric.
We compute $\Psi_4$, decompose it into spin-weighted harmonics on a geodesic sphere, and read it with kuibit.

The physics target is the quasi-normal mode $M\omega_{220} = 0.3737 - 0.0890\,i$. As you will see, the plain ADM system
becomes unstable before a clean ringdown can be measured. Finding out *why* is the point of this notebook, and it
motivates BSSN.

## Settings: where runs are written and read

This notebook writes runs to, and reads them from, **`OUTPUT_ROOT`**. By default
that is `<repository>/simulations`, the same place `pynr run par/<name>.par`
writes to from the shell. So you can:

* run here (`RUN_SIMULATIONS = True`), or
* run in a terminal, e.g. `pynr run par/schwarzschild_perturbed.par`, then set
  `RUN_SIMULATIONS = False` and only plot.

Change `OUTPUT_ROOT` to keep this notebook's runs elsewhere. For the shell to
use the same folder, run `PYNR_OUTPUT_DIR=<that folder> pynr run ...` (or
`pynr run ... --output-root <that folder>`).

In [ ]:
import os
from pynr import paths

OUTPUT_ROOT = paths.output_root()                   # default: <repo>/simulations (same as `pynr run`)
# OUTPUT_ROOT = os.path.expanduser("~/pynr_runs")   # ...or any folder you like
RUN = "schwarzschild_perturbed"                     # run name = output folder (default: parfile name)
FINAL_TIME = None                                   # e.g. 24.0 for a shorter run (~7 min on an Apple M3 Pro, 12 cores)

os.environ["PYNR_OUTPUT_DIR"] = OUTPUT_ROOT
RUN_SIMULATIONS = RUN not in paths.list_runs()      # run only if it does not exist yet; set True/False to force
print("output root  :", OUTPUT_ROOT)
print("existing runs:", paths.list_runs())
print("RUN_SIMULATIONS =", RUN_SIMULATIONS)

## Run (optional; the full parameter file takes ~10 min on an Apple M3 Pro (12 cores))

Besides the parfile output, the run samples $\gamma_{ij}$ and $K_{ij}$ on geodesic spheres of constant
coordinate radius $R$ and saves them to `sphere_samples.npz` (used in the last section). The sampler is
an ordinary schedule routine added to the `ANALYSIS` bin, the same way a thorn would add one.

In [ ]:
import numpy as np
from pynr import Simulation
from pynr.utils.geodesic import geodesic_sphere
from pynr.utils.interpolation import interpolate

SPHERE_RADII = (4.0, 6.0, 8.0)  # constant-R spheres on which gamma_ij and K_ij are sampled
SPHERE_LEVEL = 4                # geodesic refinement level: 10*4^n + 2 points (2562)
SPHERE_EVERY = 8                # sample every N iterations (dt = 0.0625 M -> every 0.5 M)


def attach_sphere_sampler(sim, radii=SPHERE_RADII, level=SPHERE_LEVEL, every=SPHERE_EVERY):
    """Schedule a routine in ANALYSIS that interpolates gamma_ij, K_ij onto geodesic spheres,
    and one in TERMINATE that saves them to <run>/sphere_samples.npz."""
    unit, tri, _ = geodesic_sphere(level)
    pts = np.concatenate([R * unit for R in radii])
    U = sim.thorn("ADMBase").U[:12]          # gxx..gzz, kxx..kzz: a contiguous view, no copy
    t, data = [], []

    def sample():
        if sim.iteration % every == 0:
            t.append(sim.time)
            data.append(interpolate(U, sim.grid.origin, sim.grid.dx, pts).reshape(12, len(radii), -1))

    def save():
        np.savez(os.path.join(sim.out_dir, "sphere_samples.npz"), t=np.array(t), radii=np.array(radii),
                 unit=unit, triangles=tri, gK=np.array(data).transpose(0, 2, 1, 3))  # (nt, nR, 12, npts)

    sim.schedule.add("ANALYSIS", sample, name="Notebook::sphere_sample")
    sim.schedule.add("TERMINATE", save, name="Notebook::sphere_save")


def run_simulation():
    over = {"IO::out_dir": RUN, "IOBasic::outInfo_every": 40}
    if FINAL_TIME is not None:
        over["Cactus::cctk_final_time"] = FINAL_TIME
    sim = Simulation.from_parfile("../par/schwarzschild_perturbed.par", overrides=over)
    attach_sphere_sampler(sim)
    sim.run()


if RUN_SIMULATIONS:
    run_simulation()

## Plot: $\Psi_4$ multipoles (read from disk with kuibit)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from kuibit.simdir import SimDir

sd = SimDir(paths.run_dir(RUN))
print("extraction radii:", sd.gws.radii)
r = min(sd.gws.radii)
for (l, m) in [(2, 0), (4, 0)]:
    p = sd.gws[r][(l, m)]
    plt.semilogy(p.t, np.abs(p.y.real), label=f"|Re Psi4_{l}{m}|")
tt = np.linspace(10, max(p.t), 50)
plt.semilogy(tt, 5e-4 * np.exp(-0.089 * (tt - 10)), "k--", label="QNM decay")
plt.xlabel("t / M"); plt.ylabel(f"|Psi4| at r = {r} M"); plt.legend()

## Checks and a (failing) ringdown fit

1. Symmetry: $\operatorname{Im}\Psi_4^{20}$ and the odd-$\ell$ modes should be at round-off level. Why?
2. Try to fit $A e^{-t/\tau}\cos(\omega t + \phi)$ after the burst and compare with $\omega = 0.3737$, $\tau = 11.2$.
   Then plot `sd.ts.norm2["H"]` on a log scale. When does the constraint violation start to grow, and how does
   that time compare with the QNM period?

In [ ]:
from scipy.optimize import curve_fit
p = sd.gws[r][(2, 0)]
t0, t1 = 12.0, 24.0      # adjust after looking at the plot above
msk = (p.t > t0) & (p.t < t1)
f = lambda t, A, tau, w, ph: A * np.exp(-(t - t0) / tau) * np.cos(w * (t - t0) + ph)
pp, _ = curve_fit(f, p.t[msk], p.y.real[msk], p0=[np.abs(p.y[msk]).max(), 11, 0.37, 0], maxfev=20000)
print(f"omega = {pp[2]:.4f}  (QNM 0.3737),  tau = {pp[1]:.2f}  (QNM 11.24)")
H = sd.ts.norm2["H"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(p.t, p.y.real); ax[0].plot(p.t[msk], f(p.t[msk], *pp), "--"); ax[0].set_title("fit")
ax[1].semilogy(H.t, H.y); ax[1].set_title("||H||_2")

### Things to try
* Compare the $(2,0)$ and $(4,0)$ modes as functions of time. Which one grows, and why is that a sign of a numerical
  rather than a physical mode?
* Change `Dissipation::epsdis`, `ADMEvolve::excision_radius` and the resolution. How do they affect the time at which $H$ starts growing?
* The perturbation does not satisfy the Hamiltonian constraint. How large is $H$ at $t=0$ compared with the unperturbed run?

## Visualizing the perturbation on spheres of constant $R$

The extraction spheres are geodesic grids (the same ones `Multipole` uses for $\Psi_4$). On a sphere of
coordinate radius $R$ with unit normal $n^i = x^i/R$ we subtract the exact Kerr-Schild background and plot

$$
\delta\gamma_{rr} = n^i n^j\left(\gamma_{ij} - \gamma^{\rm KS}_{ij}\right), \qquad
\delta K = \gamma^{ij}K_{ij} - K_{\rm KS}.
$$

Each frame is normalised by its own maximum, so the colours and the radial bulge show the **angular shape**,
and the right panel shows how big it is. Things to look for (measured at `R_PLOT = 6`):

* At $t = 0$ the sphere only sees the tail of the Gaussian shell, a pure $\ell = 2$ pattern
  ($\delta K \neq 0$ already, at the $10^{-6}$ level, because $\gamma^{ij}$ is perturbed).
* The pattern stays axisymmetric and equatorially symmetric; odd $\ell$ remain at round-off. Why?
* The burst passes $R = 6$ at $t \approx 4\text{-}10\,M$ and the $\ell = 2$ pattern oscillates in sign.
* From $t \approx 15\,M$ the amplitude grows exponentially (the constraint-violating mode). A
  **monopole** ($\ell = 0$) part grows with it: by $t = 36\,M$ it is as large as $\ell = 2$ in
  $\delta\gamma_{rr}$,, although a pure gravitational wave has no $\ell = 0$ part.
* Try `R_PLOT = 4` (on the initial shell): there $\ell = 4$ dominates $\delta K$ at the end, the
  same $\ell = 4$ growth seen in the $\Psi_4$ multipoles above.

If the run was made before the sampler existed (or from the shell), the next cell runs it again.

In [ ]:
import matplotlib.animation as animation
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import Image, display
from pynr.thorns.exact import kerr_schild

R_PLOT = 6.0                 # one of SPHERE_RADII
DEFORM = 0.15                # bulge of the sphere at the largest |value| (fraction of R)

sphere_file = os.path.join(paths.run_dir(RUN), "sphere_samples.npz")
if not os.path.exists(sphere_file):  # run made before the sampler existed, or from the shell
    run_simulation()
S = np.load(sphere_file)
iR = int(np.argmin(np.abs(S["radii"] - R_PLOT)))
R, n, tri, t = S["radii"][iR], S["unit"], S["triangles"], S["t"]
gK = S["gK"][:, iR]                                   # (nt, 12, npts)

PACK = [[0, 1, 2], [1, 3, 4], [2, 4, 5]]              # gxx gxy gxz gyy gyz gzz -> 3x3
def mat(a):                                           # (..., 6, npts) -> (..., npts, 3, 3)
    return np.moveaxis(a[..., PACK, :], -1, -3)

g, K = mat(gK[:, :6]), mat(gK[:, 6:])
g0, K0, _, _ = kerr_schild(0.0, *(R * n.T), M=1.0)    # unperturbed Kerr-Schild background
g0, K0 = mat(g0), mat(K0)

dg_rr = np.einsum("tpij,pi,pj->tp", g - g0, n, n)     # n^i n^j (gamma_ij - gamma^KS_ij)
trK = np.einsum("tpij,tpij->tp", np.linalg.inv(g), K)
dK = trK - np.einsum("pij,pij->p", np.linalg.inv(g0), K0)
fields = [(r"\delta\gamma_{rr}", dg_rr), (r"\delta K", dK)]

fig = plt.figure(figsize=(12, 4.2))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 0.05], width_ratios=[1, 1, 1.1], hspace=0.15)
axs = [fig.add_subplot(gs[0, k], projection="3d") for k in range(2)]
axt = fig.add_subplot(gs[:, 2])
for name, f in fields:
    axt.semilogy(t, np.abs(f).max(axis=1), label=rf"max $|{name}|$")
axt.set_xlabel("t / M"); axt.set_title(f"amplitude on the sphere R = {R:g} M"); axt.legend()
tline = axt.axvline(t[0], color="k", lw=0.8)
cmap = plt.get_cmap("RdBu_r")
fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(-1, 1)), cax=fig.add_subplot(gs[1, :2]),
             orientation="horizontal", label="value / max|value|  (each frame normalised separately)")

def draw(k):
    for ax, (name, f) in zip(axs, fields):
        ax.cla()
        u = f[k] / max(np.abs(f[k]).max(), 1e-300)   # normalise each frame: shows the angular shape
        v = n * (1 + DEFORM * u)[:, None]             # radial bulge proportional to the value
        col = cmap(0.5 + 0.5 * u[tri].mean(axis=1))
        ax.add_collection3d(Poly3DCollection(v[tri], facecolors=col, edgecolors=col, linewidths=0.2))
        lim = 1 + DEFORM
        ax.set(xlim=(-lim, lim), ylim=(-lim, lim), zlim=(-lim, lim))
        ax.set_box_aspect((1, 1, 1), zoom=1.5)
        ax.set_axis_off(); ax.view_init(elev=15, azim=30)
        ax.set_title(rf"${name}$  (max {np.abs(f[k]).max():.1e})")
    tline.set_xdata([t[k], t[k]])
    fig.suptitle(f"t = {t[k]:5.1f} M")

gif = os.path.join(paths.run_dir(RUN), f"sphere_R{R:g}.gif")
animation.FuncAnimation(fig, draw, frames=len(t)).save(gif, writer=animation.PillowWriter(fps=8), dpi=72)
plt.close(fig)
display(Image(filename=gif))